# Task 4 — Inference Notebook
**Heart Disease UCI Dataset**  
MLOps Assignment 01 | AIMLCZG523

Demonstrates loading the saved model and running predictions — single sample, batch, and CSV.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from data_processing import load_processed, get_feature_target, ALL_FEATURES
from predict import predict, SAMPLE_INPUT

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
sns.set_theme(style='whitegrid')

MODEL_PATH = '../models/best_model.pkl'
META_PATH  = '../models/model_meta.json'

## 1. Load Model

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

with open(META_PATH) as f:
    meta = json.load(f)

print(f'Loaded model  : {meta["best_model"]}')
print(f'Pipeline steps: {list(model.named_steps.keys())}')
print(f'Classifier    : {type(model.named_steps["classifier"]).__name__}')
print('\nModel metrics from training:')
for k, v in meta['metrics'].items():
    print(f'  {k:25s}: {v}')

## 2. Single Patient Prediction

In [ ]:
# Use the built-in sample patient
print('Input features:')
for k, v in SAMPLE_INPUT.items():
    print(f'  {k:12s}: {v}')

result = predict(SAMPLE_INPUT, model=model)
print('\nPrediction Result:')
print(json.dumps(result, indent=2))

## 3. Healthy Patient Example

In [ ]:
healthy_patient = {
    'age': 35.0, 'sex': 0.0, 'cp': 1.0, 'trestbps': 120.0, 'chol': 180.0,
    'fbs': 0.0, 'restecg': 0.0, 'thalach': 175.0, 'exang': 0.0, 'oldpeak': 0.0,
    'slope': 1.0, 'ca': 0.0, 'thal': 3.0,
}
result_healthy = predict(healthy_patient, model=model)
print('Healthy patient result:')
print(json.dumps(result_healthy, indent=2))

## 4. Batch Inference on Test Set

In [ ]:
from data_processing import prepare_train_test
df = load_processed('../data/processed/heart_disease_clean.csv')
_, X_test, _, y_test = prepare_train_test(df)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Test samples : {len(X_test)}')
print(f'Correct      : {(y_pred == y_test).sum()}')
print(f'Accuracy     : {(y_pred == y_test).mean():.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

## 5. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Inference — Confusion Matrix ({meta["best_model"]})', fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/inference_confusion_matrix.png')
plt.show()

## 6. Confidence Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_proba[y_test == 0], bins=20, alpha=0.7, color='#2ecc71', label='No Disease')
axes[0].hist(y_proba[y_test == 1], bins=20, alpha=0.7, color='#e74c3c', label='Disease')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold=0.5')
axes[0].set_title('Predicted Probability Distribution', fontweight='bold')
axes[0].set_xlabel('P(Disease)')
axes[0].set_ylabel('Count')
axes[0].legend()

max_conf = [model.predict_proba(X_test.iloc[[i]])[0].max() for i in range(len(X_test))]
axes[1].hist(max_conf, bins=20, color='#3498db', edgecolor='white')
axes[1].set_title('Model Confidence Score Distribution', fontweight='bold')
axes[1].set_xlabel('Confidence (max probability)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../screenshots/inference_confidence_dist.png')
plt.show()
print(f'Mean confidence: {np.mean(max_conf):.4f}')
print(f'Min confidence : {np.min(max_conf):.4f}')

## 7. Batch Inference from CSV

In [ ]:
# Simulate running inference on a new CSV (uses test split as example)
X_test_reset = X_test.reset_index(drop=True)
X_test_reset['predicted_class'] = model.predict(X_test)
X_test_reset['disease_probability'] = model.predict_proba(X_test)[:, 1].round(4)
X_test_reset['true_label'] = y_test.values
X_test_reset['correct'] = (X_test_reset['predicted_class'] == X_test_reset['true_label'])

output_path = '../data/processed/inference_results.csv'
X_test_reset.to_csv(output_path, index=False)
print(f'Inference results saved to {output_path}')
X_test_reset[['age','sex','cp','thalach','disease_probability','predicted_class','true_label','correct']].head(10)

## 8. API Inference via HTTP (requires running API)

In [ ]:
# Run the API first: uvicorn api.app:app --port 8080
# Then execute this cell
import requests

try:
    r = requests.post('http://localhost:8080/predict', json=SAMPLE_INPUT, timeout=3)
    print('API Response:', r.status_code)
    print(json.dumps(r.json(), indent=2))
except requests.exceptions.ConnectionError:
    print('API not running — start with: uvicorn api.app:app --host 0.0.0.0 --port 8080')